# Notebook exploring outliers using pyod

In [1]:
import platform
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "scripts" in os.getcwd():
    sys.path.append("../../..")
else:
    sys.path.append(".")

pd.set_option("display.max_columns", None)

CONTAM = 0.05
TESTNAME = "FCA"
match TESTNAME:
    case "FCA":
        from func.df_builders.dataframeBuilderFCA import get_df_column_per_combination
    case "SJT":
        from func.df_builders.dataframeBuilderSJT import get_df_column_per_combination
    case "BAQ":
        from func.df_builders.dataframeBuilderBAQ import get_df_column_per_combination
    case "PAQ":
        from func.df_builders.dataframeBuilderPAQ import get_df_column_per_combination




In [2]:
from pyod.models.copod import COPOD
from pyod.models.abod import ABOD
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
# from pyod.models.ae1svm import AE1SVM
from pyod.models.knn import KNN
from pyod.models.iforest import IForest
from pyod.models.dif import DIF
from pyod.models.lunar import LUNAR

from pyod.models.pca import PCA as pyodPCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

In [3]:
LOTS_OF_TRAINING_TIME = False # Run all
RUN_OPTIONAL = False # Just skip a few

In [4]:
def get_normal_dataframe():
    df, _, _, _ = get_df_column_per_combination(join_candidate=True)
    df.index = df["TestKey"]
    df.drop(columns="TestKey", inplace=True)
    df = df.sample(frac=1.)
    return df

In [5]:
df = get_normal_dataframe()
df

,Q117_A1,Q117_A2,Q117_A3,Q118_A1,Q118_A2,Q118_A3,Q119_A1,Q119_A2,Q119_A3,Q120_A1,Q120_A2,Q120_A3,Q121_A1,Q121_A2,Q121_A3,Q122_A1,Q122_A2,Q122_A3,Q123_A1,Q123_A2,Q123_A3,Q124_A1,Q124_A2,Q124_A3,Q125_A1,Q125_A2,Q125_A3,Q126_A1,Q126_A2,Q126_A3,Q127_A1,Q127_A2,Q127_A3,Q128_A1,Q128_A2,Q128_A3,Q129_A1,Q129_A2,Q129_A3,Q130_A1,Q130_A2,Q130_A3,Q131_A1,Q131_A2,Q131_A3,Q132_A1,Q132_A2,Q132_A3,Q133_A1,Q133_A2,Q133_A3,Q134_A1,Q134_A2,Q134_A3,Q201_A1,Q201_A2,Q201_A3,Q202_A1,Q202_A2,Q202_A3,Q203_A1,Q203_A2,Q203_A3,Q204_A1,Q204_A2,Q204_A3,Q205_A1,Q205_A2,Q205_A3,Q206_A1,Q206_A2,Q206_A3,Q207_A1,Q207_A2,Q207_A3,Q208_A1,Q208_A2,Q208_A3,Q209_A1,Q209_A2,Q209_A3,Q210_A1,Q210_A2,Q210_A3,Q211_A1,Q211_A2,Q211_A3,Q212_A1,Q212_A2,Q212_A3,Q213_A1,Q213_A2,Q213_A3,Q214_A1,Q214_A2,Q214_A3,Q215_A1,Q215_A2,Q215_A3,Q216_A1,Q216_A2,Q216_A3,Q217_A1,Q217_A2,Q217_A3,Q218_A1,Q218_A2,Q218_A3,Q220_A1,Q220_A2,Q220_A3,Q221_A1,Q221_A2,Q221_A3,Q222_A1,Q222_A2,Q222_A3,Q223_A1,Q223_A2,Q223_A3,Q224_A1,Q224_A2,Q224_A3,Q225_A1,Q225_A2,Q225_A3,Q226_A1,Q226_A2,Q226_A3,Q227_A1,Q227_A2,Q227_A3,Q228_A1,Q228_A2,Q228_A3,Q229_A1,Q229_A2,Q229_A3,Q230_A1,Q230_A2,Q230_A3,Q231_A1,Q231_A2,Q231_A3,Q232_A1,Q232_A2,Q232_A3,Q233_A1,Q233_A2,Q233_A3,Q234_A1,Q234_A2,Q234_A3,Q235_A1,Q235_A2,Q235_A3,Q236_A1,Q236_A2,Q236_A3,Q237_A1,Q237_A2,Q237_A3,Q238_A1,Q238_A2,Q238_A3,Q239_A1,Q239_A2,Q239_A3,Q240_A1,Q240_A2,Q240_A3,Q241_A1,Q241_A2,Q241_A3,Q243_A1,Q243_A2,Q243_A3,Q244_A1,Q244_A2,Q244_A3,Q245_A1,Q245_A2,Q245_A3,Q246_A1,Q246_A2,Q246_A3,Q247_A1,Q247_A2,Q247_A3,Q248_A1,Q248_A2,Q248_A3,Q249_A1,Q249_A2,Q249_A3,Q250_A1,Q250_A2,Q250_A3,Q251_A1,Q251_A2,Q251_A3,Q252_A1,Q252_A2,Q252_A3,Q253_A1,Q253_A2,Q253_A3,Q254_A1,Q254_A2,Q254_A3,Q255_A1,Q255_A2,Q255_A3,Q257_A1,Q257_A2,Q257_A3,Q258_A1,Q258_A2,Q258_A3,Q259_A1,Q259_A2,Q259_A3,Q260_A1,Q260_A2,Q260_A3,Q261_A1,Q261_A2,Q261_A3,Q262_A1,Q262_A2,Q262_A3,Q263_A1,Q263_A2,Q263_A3,Q264_A1,Q264_A2,Q264_A3,Q265_A1,Q265_A2,Q265_A3,Q266_A1,Q266_A2,Q266_A3,Q267_A1,Q267_A2,Q267_A3,Q268_A1,Q268_A2,Q268_A3,Q269_A1,Q269_A2,Q269_A3,Q270_A1,Q270_A2,Q270_A3,Q271_A1,Q271_A2,Q271_A3,Q272_A1,Q272_A2,Q272_A3,Q273_A1,Q273_A2,Q273_A3,Q290_A1,Q290_A2,Q290_A3,Q291_A1,Q291_A2,Q291_A3,Q292_A1,Q292_A2,Q292_A3,Q293_A1,Q293_A2,Q293_A3,Q294_A1,Q294_A2,Q294_A3,Q295_A1,Q295_A2,Q295_A3,Q296_A1,Q296_A2,Q296_A3,Q297_A1,Q297_A2,Q297_A3,Q298_A1,Q298_A2,Q298_A3,Q299_A1,Q299_A2,Q299_A3,Q300_A1,Q300_A2,Q300_A3,Q301_A1,Q301_A2,Q301_A3,Q302_A1,Q302_A2,Q302_A3,Q303_A1,Q303_A2,Q303_A3,Q304_A1,Q304_A2,Q304_A3,Q305_A1,Q305_A2,Q305_A3,Q331_A1,Q331_A2,Q331_A3,Q332_A1,Q332_A2,Q332_A3,Q333_A1,Q333_A2,Q333_A3,Q334_A1,Q334_A2,Q334_A3,Q335_A1,Q335_A2,Q335_A3,Q336_A1,Q336_A2,Q336_A3,Q337_A1,Q337_A2,Q337_A3,Q338_A1,Q338_A2,Q338_A3,Q339_A1,Q339_A2,Q339_A3,Q340_A1,Q340_A2,Q340_A3,Q341_A1,Q341_A2,Q341_A3,Q342_A1,Q342_A2,Q342_A3,Q343_A1,Q343_A2,Q343_A3,Q344_A1,Q344_A2,Q344_A3,Q345_A1,Q345_A2,Q345_A3,Q346_A1,Q346_A2,Q346_A3,Q347_A1,Q347_A2,Q347_A3,Q348_A1,Q348_A2,Q348_A3,Q349_A1,Q349_A2,Q349_A3,Q350_A1,Q350_A2,Q350_A3,Q351_A1,Q351_A2,Q351_A3,Q352_A1,Q352_A2,Q352_A3,Q353_A1,Q353_A2,Q353_A3,Q354_A1,Q354_A2,Q354_A3,Q355_A1,Q355_A2,Q355_A3,Q356_A1,Q356_A2,Q356_A3,Q357_A1,Q357_A2,Q357_A3,Q358_A1,Q358_A2,Q358_A3,Q359_A1,Q359_A2,Q359_A3,Q360_A1,Q360_A2,Q360_A3,Q361_A1,Q361_A2,Q361_A3,Q362_A1,Q362_A2,Q362_A3,Q363_A1,Q363_A2,Q363_A3,Q364_A1,Q364_A2,Q364_A3,Q365_A1,Q365_A2,Q365_A3,Q366_A1,Q366_A2,Q366_A3,Q367_A1,Q367_A2,Q367_A3,Q368_A1,Q368_A2,Q368_A3,Q369_A1,Q369_A2,Q369_A3,Q370_A1,Q370_A2,Q370_A3,Q371_A1,Q371_A2,Q371_A3,Q372_A1,Q372_A2,Q372_A3,Q373_A1,Q373_A2,Q373_A3,Q374_A1,Q374_A2,Q374_A3,Q375_A1,Q375_A2,Q375_A3,Q376_A1,Q376_A2,Q376_A3,Q377_A1,Q377_A2,Q377_A3,Q378_A1,Q378_A2,Q378_A3,Q379_A1,Q379_A2,Q379_A3,Q380_A1,Q380_A2,Q380_A3,Q381_A1,Q381_A2,Q381_A3,Q382_A1,Q382_A2,Q382_A3,Q383_A1,Q383_A2,Q383_A3,Q384_A1,Q384_A2,Q384_A3,Q385_A1,Q385_A2,Q385_A3,Q386_A1,Q386_A2,Q386_A3,Q387_A1,Q387_A2,Q387_A3,Q388_A1,Q388_A2,Q388_A3,Q390_A1,Q390_A2,Q390_A3,Q391_A1,Q391_A2,Q391_A3,Q392_A1,Q392_A2,Q392_A3,Q393_A1,Q393_A2,Q393_A3,Q394_A1,Q394_A2

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 160064 entries, 64033 to 34372
Columns: 694 entries, Q117_A1 to Qualification_Qualification_Vocational
dtypes: float64(660), int64(34)
memory usage: 848.7 MB


In [ ]:
def data_generator(X, batch_size=1000):
    for i in range(0, len(X), batch_size):
        yield X[i:i + batch_size]

# Functie om een model te trainen in batches
def process_batches(model, X, batch_size=1000):
    anomaly_scores = []
    
    # Train op batches
    for batch in data_generator(X, batch_size):
        model.fit(batch)  # Train op de huidige batch
        batch_scores = model.decision_function(batch)  # Verkrijg anomalie scores voor de batch
        anomaly_scores.extend(batch_scores)  # Voeg scores toe aan de lijst
    
    return np.array(anomaly_scores), model

def predict_model(model, name, prefix, X_train, contamination, df_is_anomaly):
    print(f"Training {name}!", end=" ")
    
    anomaly_scores, model = process_batches(model, X_train)

    # Bepaal anomalieën op basis van de scores (bijvoorbeeld top 5% als anomalieën)
    threshold = np.percentile(anomaly_scores, 100 - 100*contamination)  # Drempelwaarde instellen
    is_anomaly = anomaly_scores > threshold
    
    full_name = f"{prefix}_{name}"
    # df_anomaly_scores[full_name] = anomaly_scores
    df_is_anomaly[full_name] = is_anomaly.astype(int)

    # print(f"Done training  gedetecteerde anomalieën: {df_is_anomaly[full_name].sum()}")

    return "Done"

In [16]:
df_is_anomaly = pd.DataFrame(index=df.index)
# df_anomaly_scores = pd.DataFrame(index=df.index)


In [17]:
def get_X_train_normal(df):
    scaler = MinMaxScaler()
    scaler.fit(df)
    return scaler.transform(df)

def get_X_train_pca(df, n_components=10):
    X_train_scaled_normal = get_X_train_normal(df)
    pca = PCA(n_components=10)
    return pca.fit_transform(X_train_scaled_normal)


In [18]:
get_X_train_normal(df).shape, get_X_train_pca(df).shape

((160064, 694), (160064, 10))

In [19]:
# This defines all types of dataframes for the pyod ensemble 
# TODO : add pca 95% variance
# PCA(n_components=n_components_95)
# X_pca_95 = pca_95.fit_transform(X_scaled)
dataframes = {
    "normal" : lambda: get_X_train_normal(df),
    "pca_10c" : lambda: get_X_train_pca(df),
}

In [20]:
models = {
    "lof" : lambda: LOF(contamination=CONTAM, n_jobs=-1),
    "IForest" : lambda: IForest(contamination=CONTAM, n_jobs=-1),
    # "lof_manhattan" : lambda: LOF(metric='manhattan', contamination=CONTAM, n_jobs=-1),
    # "knn" : lambda: KNN(contamination=CONTAM, n_jobs=-1),
    # "IForest_555est_77samp" : lambda: IForest(contamination=CONTAM, n_estimators=555, max_samples=0.77, max_features=0.55, bootstrap=True, random_state=5, n_jobs=-1),
    # "pca" : lambda: pyodPCA(n_components=0.95, svd_solver="full", contamination=CONTAM),
    # "deepforest" : lambda: DIF(contamination=CONTAM),
    # "lunar" : lambda: LUNAR(contamination=CONTAM),
    # "abod" : lambda: ABOD(contamination=CONTAM),
    # "abod_n_neighors_10" : lambda: ABOD(contamination=CONTAM, n_neighbors=10),
    # "abod_n_neighors_20" : lambda: ABOD(contamination=CONTAM, n_neighbors=20),
}
models


{'lof': <function __main__.<lambda>()>,
 'IForest': <function __main__.<lambda>()>}

In [21]:
# Iterate one type of dataframe at a time, to preserve memory usage
for prefix, df_function in dataframes.items():
    X_train = df_function()
    print(f"Fitting {prefix} dataframe with shape: {X_train.shape}")
    # Iterate all models
    for modelname, modelfunction in models.items():
        model = modelfunction()
        start = time.time()
        try:
            predict_model(
                modelfunction(),
                name=f"{modelname}",
                prefix=prefix,
                X_train=X_train, 
                contamination=CONTAM, 
                df_is_anomaly=df_is_anomaly,
            )
        except Exception as e:
            print(f"An error occurred while executing the prediction of {modelname}: {str(e)}")
        end = time.time()
        seconds = end-start
        print(f"Took {seconds:.0f}s")
    print("="*70)

Fitting normal dataframe with shape: (160064, 694)
Training lof, using normal dataframe! Took 6s
Training IForest, using normal dataframe! Took 43s
Fitting pca_10c dataframe with shape: (160064, 10)
Training lof, using pca_10c dataframe! Took 8s
Training IForest, using pca_10c dataframe! Took 43s


In [22]:
# do_predict("lof", lambda: LOF(contamination=CONTAM, n_jobs=-1))
# do_predict("IForest", lambda: IForest(contamination=CONTAM, n_jobs=-1))
# do_predict("lof_manhattan", lambda: LOF(metric='manhattan', contamination=CONTAM, n_jobs=-1))
# do_predict("knn", lambda: KNN(contamination=CONTAM, n_jobs=-1))
# do_predict("IForest_555est_.77samp", lambda: IForest(contamination=CONTAM, n_estimators=555, max_samples=0.77, max_features=0.55, bootstrap=True, random_state=5, n_jobs=-1))
# do_predict("pca", lambda: pyodPCA(n_components=0.95, svd_solver="full", contamination=CONTAM))
# do_predict("deepforest", lambda: DIF(contamination=CONTAM))
# do_predict("lunar", lambda: LUNAR(contamination=CONTAM))
# do_predict("abod", lambda: ABOD(contamination=CONTAM))
# do_predict("abod_n_neighors_10", lambda: ABOD(contamination=CONTAM, n_neighbors=10))
# do_predict("abod_n_neighors_20", ABOD(contamination=CONTAM, n_neighbors=30))


In [ ]:
df_is_anomaly

,normal_lof_0.05,pca_lof_0.05,normal_lof_manhattan_0.05,pca_lof_manhattan_0.05,normal_knn_0.05,pca_knn_0.05,normal_IForest_0.05,pca_IForest_0.05,normal_IForest_555est_.77samp_0.05,pca_IForest_555est_.77samp_0.05,normal_deepforest_0.05,pca_deepforest_0.05
TestKey,,,,,,,,,,,,
173930,1,0,0,0,1,0,0,0,1,0,0,1
168941,0,0,0,0,0,0,0,0,0,0,0,0
168707,0,0,0,0,0,0,0,0,0,0,0,0
167779,0,0,0,0,0,0,0,0,0,0,0,0
176234,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
179708,0,0,0,0,0,0,0,0,0,0,0,0
175749,0,0,0,0,0,0,0,0,0,0,0,0
172863,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
def compare_models(df):
    columns = df.columns.to_list()
    columns_copy = columns.copy()
    for col in columns:
        columns_copy.remove(col)
        for other in columns_copy:
            similarity = (df[col] == df[other]).sum() / len(df) * 100
            print(f"{similarity:.2f}% = {col} - {other} ")
        print("*"*70)

compare_models(df_is_anomaly)

93.20% = normal_lof_0.05 - pca_lof_0.05 
96.26% = normal_lof_0.05 - normal_lof_manhattan_0.05 
93.19% = normal_lof_0.05 - pca_lof_manhattan_0.05 
97.18% = normal_lof_0.05 - normal_knn_0.05 
92.63% = normal_lof_0.05 - pca_knn_0.05 
96.71% = normal_lof_0.05 - normal_IForest_0.05 
91.91% = normal_lof_0.05 - pca_IForest_0.05 
97.04% = normal_lof_0.05 - normal_IForest_555est_.77samp_0.05 
92.00% = normal_lof_0.05 - pca_IForest_555est_.77samp_0.05 
94.11% = normal_lof_0.05 - normal_deepforest_0.05 
92.29% = normal_lof_0.05 - pca_deepforest_0.05 
**********************************************************************
92.51% = pca_lof_0.05 - normal_lof_manhattan_0.05 
98.33% = pca_lof_0.05 - pca_lof_manhattan_0.05 
93.00% = pca_lof_0.05 - normal_knn_0.05 
95.19% = pca_lof_0.05 - pca_knn_0.05 
93.15% = pca_lof_0.05 - normal_IForest_0.05 
93.80% = pca_lof_0.05 - pca_IForest_0.05 
93.18% = pca_lof_0.05 - normal_IForest_555est_.77samp_0.05 
94.24% = pca_lof_0.05 - pca_IForest_555est_.77samp_0.05 
9

In [ ]:
df_outliers = df_is_anomaly
columns = df_outliers.columns.to_list()
# columns.remove("TestKey")
df_outliers['count'] =  df_outliers[columns].sum(axis=1)
df_outliers['percentage'] = df_outliers['count'] / len(columns)
df_outliers

,normal_lof,normal_IForest,pca_10c_lof,pca_10c_IForest,count,pyod % outlier
TestKey,,,,,,
64033,0,0,0,0,0,0.0
82093,0,0,0,0,0,0.0
101020,0,0,0,0,0,0.0
144003,0,0,0,0,0,0.0
126900,0,0,0,0,0,0.0
...,...,...,...,...,...,...
147101,0,0,0,0,0,0.0
73097,0,0,0,0,0,0.0
89219,0,0,0,0,0,0.0


In [24]:
df_outliers['count'].value_counts()

count
0    138114
1     15249
2      3452
3      3133
4       116
Name: count, dtype: int64

In [ ]:
TRESHOLD = 0.5
df_outliers["is_anomaly"] = np.where(df_outliers['percentage'] >= TRESHOLD, 1, 0)
df_outliers

,normal_lof,normal_IForest,pca_10c_lof,pca_10c_IForest,count,pyod % outlier,is_anomaly
TestKey,,,,,,,
64033,0,0,0,0,0,0.0,0
82093,0,0,0,0,0,0.0,0
101020,0,0,0,0,0,0.0,0
144003,0,0,0,0,0,0.0,0
126900,0,0,0,0,0,0.0,0
...,...,...,...,...,...,...,...
147101,0,0,0,0,0,0.0,0
73097,0,0,0,0,0,0.0,0
89219,0,0,0,0,0,0.0,0


In [27]:
csv_path = os.path.join("/home/miked/code/dep2-g2/anomalies", TESTNAME, "csv", f"pyod_ensemble_{TESTNAME}.csv")
df_outliers.to_csv(csv_path, index=True)

In [28]:
df_outliers["is_anomaly"].sum()

np.int64(6701)